In [1]:
using Pkg
Pkg.activate(joinpath(@__DIR__, "..", "..", ".."))  # go up three levels (from PauliSampling.jl\numerics\quantumboltzmannmachine\ceofficient_truncation to to PauliSampling.jl)

  Activating project at `~/Desktop/PauliSampling.jl`


In [2]:
# Load all packages
using PauliSampling, PauliPropagation
using LinearAlgebra, Statistics, CSV, DataFrames, Plots, ProgressMeter
using Base.Threads

In [3]:
# --- Setup Definitions ---

struct SimConfig
    nq::Int
    betas::Vector{Float64}
    alphas::Vector{Float64}
    num_samples::Int
    model_type::Symbol 
end

function setup_hamiltonian(config::SimConfig)
    # File: src/QuantumBoltzmannMachine/hamiltonians.jl
    # lat = SquareLattice(4, 3) 
    lat = ChainLattice(config.nq)
    params = parameters(config.model_type, lat.n; init=:zeros,
                            coupling_vals=Dict(:XX => 1.0, :YY => 1.0, :ZZ => 1.0))
    return makehamiltonian(params, lat; connectivity=:nearest, periodic=false, model=config.model_type)
end

function compute_tvd(p1::Vector{Float64}, p2::Vector{Float64})
    # Equation (12) from the Draft Paper
    return 0.5 * sum(abs.(p1 .- p2))
end

function get_probabilities(rho, num_samples)
    nq = rho.nqubits
    counts = zeros(Int, 2^nq)
    # Using locally normalized conditional sampling (Algorithm 1)
    for _ in 1:num_samples
        bs = sample_bitstring(rho; prob_method=:approx)
        idx = foldl((acc, b) -> 2acc + b, reverse(bs), init=0) + 1
        counts[idx] += 1
    end
    return counts ./ num_samples
end

function run_experiment(config::SimConfig)
    H = setup_hamiltonian(config)
    
    # Thread-safe storage for results
    results_lock = ReentrantLock()
    all_results = Tuple{Float64, Float64, Float64}[]
    
    # Progress monitoring that works across threads
    total_steps = length(config.betas) * length(config.alphas)
    prog = Progress(total_steps, desc="Parallel Simulation:")

    for β in config.betas
        println("\nStatus: Starting experiments for Beta: $β")
        flush(stdout) # Force the terminal to print immediately
        
        # 1. Prepare evolution once per Beta (Equation 29 in the paper)
        circ, θ = paulistringtocircuit(H * β)
        
        # 2. Reference State calculation (Done sequentially to save memory)
        rho_exact = makethermalstate(config.nq, circ, θ, config.nq*10; 
                                     max_weight = config.nq,
                                     min_abs_coeff=1e-10, optimize_for_z_basis=true)
        p_exact = get_probabilities(rho_exact, config.num_samples)

        # 3. Parallelize the Truncation Sweep (Independent Alphas)
        # Each alpha uses a different truncation threshold as discussed in Section III
        Threads.@threads for α in config.alphas
            # Locally normalized conditional sampling (Algorithm 1)
            rho_trunc = makethermalstate(config.nq, circ, θ, config.nq*10; 
                                         min_abs_coeff=α, optimize_for_z_basis=true)
            
            p_trunc = get_probabilities(rho_trunc, config.num_samples)
            dist = compute_tvd(p_exact, p_trunc)
            
            # Thread-safe result collection
            lock(results_lock) do
                push!(all_results, (β, α, dist))
                next!(prog) # Update progress bar
            end
        end
    end
    
    # Construct final DataFrame and sort for plotting
    df = DataFrame(all_results, [:beta, :alpha, :tvd])
    return sort!(df, [:beta, :alpha])
end

run_experiment (generic function with 1 method)

In [17]:
# --- Execution ---

# Configuration for nq=6
config = SimConfig(
    6,                          # nq
    [0.5, 5.0, 40.0],           # Low, Medium, High Beta
    [2.0^-i for i in 2:0.1:32], # Finer alpha step for a smoother plot
    1_000_000,                  # Samples (Solid statistical resolution for nq=6)
    :heisenberg
)

# Run Simulation
results_df = run_experiment(config)


Status: Starting experiments for Beta: 0.5


Parallel Simulation:  33%|██████████▍                    |  ETA: 0:01:54


Status: Starting experiments for Beta: 5.0


Parallel Simulation:  66%|████████████████████▌          |  ETA: 0:00:58


Status: Starting experiments for Beta: 40.0


Parallel Simulation: 100%|███████████████████████████████| Time: 0:02:55


Row,beta,alpha,tvd
,Float64,Float64,Float64
1,0.5,2.32831e-10,0.004217
2,0.5,2.49542e-10,0.004522
3,0.5,2.67452e-10,0.003782
4,0.5,2.86648e-10,0.003888
5,0.5,3.07222e-10,0.00417
6,0.5,3.29272e-10,0.004702
7,0.5,3.52905e-10,0.004705
8,0.5,3.78234e-10,0.004463
9,0.5,4.05382e-10,0.004259


In [18]:
# Save Results to CSV
# Cite: Data useful for publication reproducibility
CSV.write("truncation_tvd_$(config.model_type)_nq$(config.nq)_sampling.csv", results_df)
@info "Results saved to $filename"

UndefVarError: UndefVarError: `filename` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
Hint: a global variable of this name may be made accessible by importing FilePathsBase in the current active module Main

## exact dsitributions

In [4]:
"""
    get_exact_distribution(rho)

Replaces sampling-based counting. Iterates through all 2^N bitstrings 
and computes either the exact Born probability or the heuristic 
probability p_hat(x) based on the state's Diagonal Pauli Expansion.
"""
function get_exact_distribution(rho; method=:approx)
    nq = rho.nqubits
    num_states = 1 << nq
    probs = zeros(Float64, num_states)
    
    # Pre-generate all possible bitstrings for the 2^N space
    for i in 0:(num_states - 1)
        # Convert integer index i to BitVector (matching the library order)
        bs = BitVector(undef, nq)
        for q in 1:nq
            bs[nq + 1 - q] = (i >> (q - 1)) & 1 == 1
        end
        
        # Select the deterministic calculation method
        if method == :exact
            # For the target distribution p(x)
            probs[i + 1] = PauliSampling.get_exact_prob(rho, bs)
        else
            # For the truncated heuristic distribution p_hat(x) 
            # produced by Algorithm 1 logic
            probs[i + 1] = PauliSampling.get_approx_prob(rho, bs)
        end
    end
    return probs
end

function run_experiment_exact(config::SimConfig)
    H = setup_hamiltonian(config)
    results_lock = ReentrantLock()
    all_results = Tuple{Float64, Float64, Float64}[]
    
    total_steps = length(config.betas) * length(config.alphas)
    prog = Progress(total_steps, desc="Exact TVD Simulation:")

    for β in config.betas
        println("\nStatus: Analyzing Beta: $β")
        flush(stdout)
        
        # Prepare the reference state once per Beta
        circ, θ = paulistringtocircuit(H * β)
        
        # Calculate Reference Distribution Exactly (No Sampling)
        # We use :exact here to get the true Born probabilities p(x)
        rho_ref = makethermalstate(config.nq, circ, θ, config.nq*10; 
                                   max_weight = config.nq,
                                   min_abs_coeff=1e-10, optimize_for_z_basis=true)
        p_ref = get_exact_distribution(rho_ref; method=:exact)

        # Parallelize the Truncation Sweep
        Threads.@threads for α in config.alphas
            # Truncated state according to Section III
            rho_trunc = makethermalstate(config.nq, circ, θ, config.nq*10; 
                                         min_abs_coeff=α, optimize_for_z_basis=true)
            
            # Calculate Heuristic Distribution Exactly
            # We use :approx here to mimic the Algorithm 1 logic without noise
            p_trunc = get_exact_distribution(rho_trunc; method=:approx)
            
            dist = compute_tvd(p_ref, p_trunc) # Equation 12
            
            lock(results_lock) do
                push!(all_results, (β, α, dist))
                next!(prog)
            end
        end
    end
    
    df = DataFrame(all_results, [:beta, :alpha, :tvd])
    return sort!(df, [:beta, :alpha])
end

run_experiment_exact (generic function with 1 method)

In [15]:
# --- Execution ---

# Configuration for nq=6
config = SimConfig(
    6,                          # nq
    [0.5, 5.0, 40.0],           # Low, Medium, High Beta
    [2.0^-i for i in 2:0.1:32], # Finer alpha step for a smoother plot
    1_000_000,                  # Samples (Solid statistical resolution for nq=6)
    :heisenberg
)

# Run Simulation
results_df = run_experiment_exact(config)


Status: Analyzing Beta: 0.5


Exact TVD Simulation:  33%|██████████                    |  ETA: 0:00:02


Status: Analyzing Beta: 5.0


Exact TVD Simulation:  64%|███████████████████▎          |  ETA: 0:00:01


Status: Analyzing Beta: 40.0


Exact TVD Simulation: 100%|██████████████████████████████| Time: 0:00:03


Row,beta,alpha,tvd
,Float64,Float64,Float64
1,0.5,2.32831e-10,1.52297e-10
2,0.5,2.49542e-10,1.52528e-10
3,0.5,2.67452e-10,1.51483e-10
4,0.5,2.86648e-10,1.51401e-10
5,0.5,3.07222e-10,1.51347e-10
6,0.5,3.29272e-10,1.50562e-10
7,0.5,3.52905e-10,2.40516e-10
8,0.5,3.78234e-10,2.4056e-10
9,0.5,4.05382e-10,2.42326e-10


In [16]:
# Save Results to CSV
# Cite: Data useful for publication reproducibility
CSV.write("truncation_tvd_$(config.model_type)_nq$(config.nq)_exact.csv", results_df)
@info "Results saved to $filename"

UndefVarError: UndefVarError: `filename` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
Hint: a global variable of this name may be made accessible by importing FilePathsBase in the current active module Main

### bigger system

In [5]:
# --- Execution ---

# Configuration for nq=6
config = SimConfig(
    12,                          # nq
    [0.1, 1, 5.0],           # Low, Medium, High Beta
    [2.0^-i for i in 2:0.1:32], # Finer alpha step for a smoother plot
    1_000_000,                  # Samples (Solid statistical resolution for nq=6)
    :heisenberg
)

# Run Simulation
results_df = run_experiment_exact(config)


Status: Analyzing Beta: 0.1


Exact TVD Simulation:  33%|██████████                    |  ETA: 0:03:24


Status: Analyzing Beta: 1.0


Exact TVD Simulation:  67%|████████████████████          |  ETA: 0:35:08


Status: Analyzing Beta: 5.0


Exact TVD Simulation: 100%|██████████████████████████████| Time: 6:23:42


Row,beta,alpha,tvd
,Float64,Float64,Float64
1,0.1,2.32831e-10,4.8129e-8
2,0.1,2.49542e-10,5.45256e-8
3,0.1,2.67452e-10,6.09798e-8
4,0.1,2.86648e-10,7.03765e-8
5,0.1,3.07222e-10,7.99466e-8
6,0.1,3.29272e-10,8.78621e-8
7,0.1,3.52905e-10,9.837e-8
8,0.1,3.78234e-10,1.10582e-7
9,0.1,4.05382e-10,1.23171e-7


In [6]:
# Save Results to CSV
# Cite: Data useful for publication reproducibility
CSV.write("truncation_tvd_$(config.model_type)_nq$(config.nq)_exact.csv", results_df)
@info "Results saved to $filename"

UndefVarError: UndefVarError: `filename` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
Hint: a global variable of this name may be made accessible by importing FilePathsBase in the current active module Main

# for plot 2

In [14]:
using PauliSampling, PauliPropagation, DataFrames, CSV

function save_3d_hist_data(config::SimConfig, target_beta::Float64, target_alphas::Vector{Float64})
    H = setup_hamiltonian(config)
    circ, θ = paulistringtocircuit(H * target_beta)
    
    # 1. Reference State (Almost Exact)
    rho_exact = makethermalstate(config.nq, circ, θ, config.nq*10; 
                                 min_abs_coeff=1e-10, optimize_for_z_basis=true)
    p_exact = get_probabilities(rho_exact, config.num_samples)

    df = DataFrame(bitstring = 0:(2^config.nq - 1), exact = p_exact)

    # 2. Sequential Alphas
    for (i, α) in enumerate(target_alphas)
        @info "Generating for Alpha index $i: $α"
        rho_trunc = makethermalstate(config.nq, circ, θ, config.nq*10; 
                                     min_abs_coeff=α, optimize_for_z_basis=true)
        p_trunc = get_probabilities(rho_trunc, config.num_samples)
        # Name columns by exponent to avoid parsing issues in Python
        exponent = Int(abs(log2(α)))
        df[!, Symbol("log2alpha_neg$(exponent)")] = p_trunc
    end

    CSV.write("hist3d_data_beta$(target_beta).csv", df)
    println("Data saved for 3D plotting.")
end

# Execution for 6 qubits
config = SimConfig(6, [5.0], [], 1_000_000, :heisenberg)
# Selecting 4 alphas: one very small (near exact) and three larger truncations
target_alphas = [2.0^-10, 2.0^-8, 2.0^-6, 2.0^-4]
save_3d_hist_data(config, 5.0, target_alphas)

┌ Info: Generating for Alpha index 1: 0.0009765625
└ @ Main /Users/micheleminervini/Desktop/PauliSampling.jl/numerics/paper_figures/figure_2/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y225sZmlsZQ==.jl:16
┌ Info: Generating for Alpha index 2: 0.00390625
└ @ Main /Users/micheleminervini/Desktop/PauliSampling.jl/numerics/paper_figures/figure_2/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y225sZmlsZQ==.jl:16
┌ Info: Generating for Alpha index 3: 0.015625
└ @ Main /Users/micheleminervini/Desktop/PauliSampling.jl/numerics/paper_figures/figure_2/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y225sZmlsZQ==.jl:16
┌ Info: Generating for Alpha index 4: 0.0625
└ @ Main /Users/micheleminervini/Desktop/PauliSampling.jl/numerics/paper_figures/figure_2/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y225sZmlsZQ==.jl:16


Data saved for 3D plotting.


In [21]:
nq = 6

num_layers_trotter = nq * 10
# 1. Define the 1D Lattice geometry
lat_2d = SquareLattice(3, 2)
# 2. Initialize parameters
params = parameters(:heisenberg, lat_2d.n; 
                     init=:zeros, 
                     coupling_vals=Dict(:XX => 1.0, :YY => 1.0, :ZZ => 1.0))
# 3. Build the Hamiltonian Matrix
H_paulis = makehamiltonian(params, lat_2d; 
                            connectivity=:nearest, 
                            periodic=false,
                            model=:heisenberg);

In [22]:
low_beta = 0.5;

circuit_low_beta, theta_low_beta = paulistringtocircuit(H_paulis * low_beta);

In [29]:
rho_highT_exact = makethermalstate(nq, circuit_low_beta, theta_low_beta, num_layers_trotter;
                            max_weight=nq, min_abs_coeff=1e-10, optimize_for_z_basis=true)

PauliSum(nqubits: 6, 32 Pauli terms:
 0.00049513 * IZZIII
 0.00029846 * IZZZZI
 -0.00010247 * ZIIZZZ
 0.00030311 * ZZZIZI
 -0.0021376 * IIZIZI
 -9.9345e-5 * IZZZIZ
 0.00030355 * ZZIIZZ
 0.00049513 * ZIIZII
 0.015625 * IIIIII
 0.00025878 * IZIIIZ
 -9.5206e-5 * ZIIIIZ
 0.00030311 * ZZIZIZ
 -0.0021376 * IIIZIZ
 -0.00012268 * ZZZZZZ
 -0.00010249 * ZZZIIZ
 0.00058749 * IIZZZZ
 -9.5206e-5 * IZIIZI
 0.00030304 * ZIZIZZ
 -0.0021382 * IZIZII
 -0.0021467 * IIZZII
  ⋮)

In [25]:
high_beta = 50;

circuit_high_beta, theta_high_beta = paulistringtocircuit(H_paulis * high_beta);

In [30]:
rho_lowT_exact = makethermalstate(nq, circuit_high_beta, theta_high_beta, num_layers_trotter;
                            max_weight=nq, min_abs_coeff=1e-10, optimize_for_z_basis=true)

PauliSum(nqubits: 6, 32 Pauli terms:
 0.0048173 * IZZIII
 0.0036743 * IZZZZI
 -0.0048173 * ZIIZZZ
 0.0076647 * ZZZIZI
 -0.0076647 * IIZIZI
 -0.0033385 * IZZZIZ
 0.0090751 * ZZIIZZ
 0.0048173 * ZIIZII
 0.015625 * IIIIII
 0.0033385 * IZIIIZ
 -0.0036743 * ZIIIIZ
 0.0076647 * ZZIZIZ
 -0.0076647 * IIIZIZ
 -0.015625 * ZZZZZZ
 -0.0046523 * ZZZIIZ
 0.011752 * IIZZZZ
 -0.0036743 * IZIIZI
 0.0083548 * ZIZIZZ
 -0.0083548 * IZIZII
 -0.0090751 * IIZZII
  ⋮)